In [2]:
from pathlib import Path
import pandas as pd

from tower_eval.utils import write_lines

# Download data

In [63]:
maia_lps = ["en-de", "en-fr", "en-pt-br"]
download_data_dir = Path("downloaded_data/")
download_data_dir.mkdir(exist_ok=True)
for lp in maia_lps:
    url = f"https://raw.githubusercontent.com/Unbabel/MAIA/master/data/MAIA.{lp}.csv"
    df = pd.read_csv(url, keep_default_na=False, na_values=["NaN"])
    df.to_csv(download_data_dir / f"MAIA.{lp}.csv", index=False)

# Preprocess data

## Create raw data per segment

In [64]:
raw_data_dir = Path("raw_data/")
for maia_lp in maia_lps:
    if maia_lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = maia_lp.split("-")
    df = pd.read_csv(
        download_data_dir / f"MAIA.{maia_lp}.csv",
        keep_default_na=False,
        na_values=["NaN"],
    )
    # rename for tower_eval purposes
    df.rename(
        columns={
            "source_segment": "src",
            "pe_segment": "ref",
        },
        inplace=True,
    )
    # en-xx direction
    en_xx_df = df[df["source_language"] == "en"].reset_index(drop=True)
    save_dir = Path(f"{raw_data_dir}/mt/maia.{eng}-{non_eng}")
    save_dir.mkdir(parents=True, exist_ok=True)
    en_xx_df.to_json(
        save_dir / f"test.jsonl", index=False, orient="records", lines=True
    )
    # xx-en direction
    xx_en_df = df[df["target_language"] == "en"].reset_index(drop=True)
    save_dir = Path(f"{raw_data_dir}/mt/maia.{non_eng}-{eng}")
    save_dir.mkdir(parents=True, exist_ok=True)
    xx_en_df.to_json(
        save_dir / f"test.jsonl", index=False, orient="records", lines=True
    )

## Create tower instructions

In [65]:
from transformers import AutoTokenizer

In [66]:
t = AutoTokenizer.from_pretrained("Unbabel/TowerInstruct-13B-v0.2")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


### No context

In [67]:
def apply_instruction(src, src_lang, tgt_lang, tokenizer):
    messages = [
        {
            "role": "user",
            "content": f"Translate the following {src_lang} source text to {tgt_lang}:\n{src_lang}: {src}\n{tgt_lang}: ",
        }
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

In [68]:
instructions_data_dir = Path("instructions/no_context")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
}

for maia_lp in maia_lps:
    if maia_lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = maia_lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/maia.{eng}-{non_eng}")
    df = pd.read_json(raw_data_dir / f"mt/maia.{eng}-{non_eng}/test.jsonl", lines=True)
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[eng], code_lang_dict[non_eng], t
            )
        )
        .tolist()
    )
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/maia.{non_eng}-{eng}")
    df = pd.read_json(raw_data_dir / f"mt/maia.{non_eng}-{eng}/test.jsonl", lines=True)
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t
            )
        )
        .tolist()
    )
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)

In [72]:
type(
    pd.read_json(
        "/mnt/data/jpombal/wmt24_chat_translation/raw_data/mt/maia.fr-en/test.jsonl",
        lines=True,
    ).loc[2671]["mt_segment"]
)

str

### Full context

In [ ]:
def df_to_chat_instructions(df, src_lang, tgt_lang, tokenizer):


In [ ]:
instructions_data_dir = Path("instructions/full_context")
code_lang_dict = {
    "en": "English",
    "de": "German",
    "fr": "French",
    "pt-br": "Brazilian Portuguese",
}

for maia_lp in maia_lps:
    if maia_lp == "en-pt-br":
        non_eng = "pt-br"
        eng = "en"
    else:
        eng, non_eng = maia_lp.split("-")

    # en-xx direction
    save_dir = Path(f"{instructions_data_dir}/mt/maia.{eng}-{non_eng}")
    df = pd.read_json(raw_data_dir / f"mt/maia.{eng}-{non_eng}/test.jsonl", lines=True)
    instructions = df_to_chat_instructions(df, eng, non_eng, t)
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)
    # xx-en direction
    save_dir = Path(f"{instructions_data_dir}/mt/maia.{non_eng}-{eng}")
    df = pd.read_json(raw_data_dir / f"mt/maia.{non_eng}-{eng}/test.jsonl", lines=True)
    instructions = (
        df["src"]
        .apply(
            lambda x: apply_instruction(
                x, code_lang_dict[non_eng], code_lang_dict[eng], t
            )
        )
        .tolist()
    )
    write_lines(save_dir / "instructions.txt", instructions, escape_newline=True)

In [3]:
df = pd.read_json(
    "/mnt/data/jpombal/wmt24_chat_translation/raw_data/mt/maia.fr-en/test.jsonl",
    lines=True,
)

In [8]:
df = pd.read_csv(
    "/mnt/data/jpombal/wmt24_chat_translation/downloaded_data/MAIA.en-fr.csv"
)

In [10]:
df[df["doc_id"] == "en_fr_#CLIENT-02#_default020-12-20-35"]

,doc_id,content_type,source_language,target_language,client,brand,tone,source_segment,mt_segment,pe_segment,translation_direction
0,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,"Bonjour,","Hello,","Hello,",customer
1,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Mes livres #PRS_ORG# empruntés restent toujour...,My borrowed #PRS_ORG# books are still protecte...,My borrowed #PRS_ORG# books are still protecte...,customer
2,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Ça ne fonctionne pas et ça fait plusieurs fois...,It doesn't work and it's been several times si...,It doesn't work and I've been through this pro...,customer
3,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Merci.,Thank you.,Thank you.,customer
4,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,#EMAIL#,#EMAIL#,#EMAIL#,customer
5,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Je n'en possède pas et en n'Ai pas acheté,I don't own it and I don't buy it,I don't own it and I didn't buy it.,customer
6,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Allo??,Hello?,Hello?,customer
7,en_fr_#CLIENT-02#_default020-12-20-35,chat,en,fr,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Hello and thank you.,Bonjour et merci,Bonjour et merci.,agent
8,en_fr_#CLIENT-02#_default020-12-20-35,chat,fr,en,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Bonjour :),Hello,Hello :),customer
9,en_fr_#CLIENT-02#_default020-12-20-35,chat,en,fr,#CLIENT-02#,default-brand-of-#CLIENT-02#,formal,Are you readiung a message?,Recevez-vous un message ?,Lisez-vous un message ?,agent
